In [236]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_auc_score, f1_score, roc_curve, classification_report, 
                             cohen_kappa_score, make_scorer)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import seaborn as sns
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.svm import SVC

In [557]:
train_data = pd.read_csv("CAH-201803-train.csv")
test_data = pd.read_csv("CAH-201803-test.csv")

In [238]:
X = train_data.drop(["political_affiliation", "id_num"], axis = 1)
y = train_data["political_affiliation"]

In [550]:

def get_metrics(X, y, model_type):
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y)

    # Initialize model and params
    if model_type == "LDA":
        model = LinearDiscriminantAnalysis()
        params = {"model__solver": ['svd', 'lsqr']}
    elif model_type == "QDA":
        model = QuadraticDiscriminantAnalysis()
        params = {'model__reg_param': [0.0, 0.1, 0.2, 0.5, 1.0]}
    elif model_type == "SVC":
        model = SVC()
        params = {'model__kernel': ['linear', 'poly', 'rbf'], 'model__C': [0.1, 1, 10], 
                  'model__gamma': ['scale', 'auto']}
    elif model_type == "SVM":
        model = SVC()
        params = {'model__kernel': ['poly'], 'model__C': [0.1, 1, 10], 'model__degree': [3, 4, 5]}
    elif model_type == "KNN":
        model = KNeighborsClassifier()
        params = {
            'model__n_neighbors': range(1, 20),
            #'model__n_neighbors': [10],
            'model__weights': ['uniform', 'distance'], 
            'model__metric': ['euclidean', 'manhattan', 'minkowski']
        }
    elif model_type == "log":
        model = LogisticRegression()
        params = {'model__C': [0.01, 0.1, 1, 10, 100]}
    # Define preprocessing pipeline
    ct = ColumnTransformer([
    ("dummify", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
     make_column_selector(dtype_include=object)),
    ("standardize", StandardScaler(), make_column_selector(dtype_include=np.number))
],
remainder="passthrough").set_output(transform="pandas")

    # Build pipeline with column transformer and model
    pipeline = Pipeline([
        ('preprocessor', ct),
        ('model', model)
    ])

    # Perform Grid Search for hyperparameter tuning
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, scoring='accuracy', n_jobs = -1)
    grid_search.fit(X_train, y_train)

    # Get the best model and best parameters
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"Best Model: {best_params}")

    # Cross-validation to get Accuracy
    cv_accuracy = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy').mean()
    print(f"Cross-validated Accuracy: {cv_accuracy:.4f}")

    # Fit the final model on the whole training set
    best_model.fit(X, y)

    # Predict on the test set and calculate confusion matrix
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print("Class Order:", best_model.classes_)
    print("Confusion Matrix:")
    print(cm)

    return best_model

In [551]:
final_model_fit = get_metrics(X, y, model_type = "KNN")

Best Model: {'model__metric': 'euclidean', 'model__n_neighbors': 11, 'model__weights': 'distance'}
Cross-validated Accuracy: 0.6000
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[12  0  0]
 [ 0 11  0]
 [ 0  0 11]]
Best Model: {'model__metric': 'euclidean', 'model__n_neighbors': 11, 'model__weights': 'distance'}
Cross-validated Accuracy: 0.6000
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[12  0  0]
 [ 0 11  0]
 [ 0  0 11]]


In [556]:
final_predictions = pd.DataFrame(
    {"id_num": test_data['id_num'],
    "political_affiliation_predicted": final_model_fit.predict(test_data)}
)


In [560]:
final_predictions.to_csv('final_predictions1.csv', index = False)


In [562]:
get_metrics(X, y, model_type = "SVM")
get_metrics(X, y, model_type = "log")
get_metrics(X, y, model_type = "SVC")
get_metrics(X, y, model_type = "LDA")
get_metrics(X, y, model_type = "QDA")

Best Model: {'model__C': 1, 'model__degree': 3, 'model__kernel': 'poly'}
Cross-validated Accuracy: 0.5111
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[12  0  0]
 [ 1 10  0]
 [ 1  0 10]]
Best Model: {'model__C': 1, 'model__degree': 3, 'model__kernel': 'poly'}
Cross-validated Accuracy: 0.5111
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[12  0  0]
 [ 1 10  0]
 [ 1  0 10]]


Best Model: {'model__C': 0.01}
Cross-validated Accuracy: 0.5556
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[9 2 1]
 [5 3 3]
 [3 0 8]]
Best Model: {'model__C': 0.01}
Cross-validated Accuracy: 0.5556
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[9 2 1]
 [5 3 3]
 [3 0 8]]


Best Model: {'model__C': 10, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
Cross-validated Accuracy: 0.6222
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[ 9  3  0]
 [ 1 10  0]
 [ 0  1 10]]
Best Model: {'model__solver': 'svd'}
Best Model: {'model__C': 10, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
Cross-validated Accuracy: 0.6222
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[ 9  3  0]
 [ 1 10  0]
 [ 0  1 10]]
Best Model: {'model__solver': 'svd'}


Cross-validated Accuracy: 0.5630
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[ 9  3  0]
 [ 1  9  1]
 [ 0  1 10]]
Best Model: {'model__reg_param': 0.1}
Cross-validated Accuracy: 0.5111
Cross-validated Accuracy: 0.5630
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[ 9  3  0]
 [ 1  9  1]
 [ 0  1 10]]
Best Model: {'model__reg_param': 0.1}
Cross-validated Accuracy: 0.5111


Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[10  1  1]
 [ 1 10  0]
 [ 0  0 11]]
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[10  1  1]
 [ 1 10  0]
 [ 0  0 11]]


/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonPro

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('dummify',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7f80f892baf0>),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7f80f8928040>)])),
                ('model', QuadraticDiscriminantAnalysis(reg_param=0.1))])

In [593]:

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# Preprocessing: Encoding and Scaling
ct = ColumnTransformer([
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"), make_column_selector(dtype_include="object")),
    ("scaler", StandardScaler(), make_column_selector(dtype_include="number"))
], remainder="passthrough")

# Define SVC and parameters for Grid Search
svc_model = SVC()
svc_params = {
    "classifier__kernel": ["linear", "rbf", "poly"],
    "classifier__C": [.1],
    "classifier__gamma": ["scale", "auto"]
}

# Pipeline for SVC
svc_pipeline = Pipeline([
    ("preprocessor", ct),
    ("classifier", svc_model)
])

# Grid Search with Cross-Validation
grid_search = GridSearchCV(svc_pipeline, param_grid=svc_params, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model and Parameters
print(f"Best Parameters: {grid_search.best_params_}")
best_model = grid_search.best_estimator_

# Cross-Validation Accuracy
cv_accuracy = cross_val_score(best_model, X_train, y_train, cv=5, scoring="accuracy").mean()
print(f"Cross-Validated Accuracy: {cv_accuracy:.4f}")

# Evaluate on Test Set
y_pred = best_model.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Best Parameters: {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
Cross-Validated Accuracy: 0.5852

Classification Report:
               precision    recall  f1-score   support

    Democrat       0.75      0.50      0.60        12
 Independent       0.53      0.73      0.62        11
  Republican       0.73      0.73      0.73        11

    accuracy                           0.65        34
   macro avg       0.67      0.65      0.65        34
weighted avg       0.67      0.65      0.65        34

Confusion Matrix:
 [[6 6 0]
 [0 8 3]
 [2 1 8]]
Best Parameters: {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
Cross-Validated Accuracy: 0.5852

Classification Report:
               precision    recall  f1-score   support

    Democrat       0.75      0.50      0.60        12
 Independent       0.53      0.73      0.62        11
  Republican       0.73      0.73      0.73        11

    accuracy                      

In [604]:
def get_metrics(X, y, model_type):

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

    # Initialize model and params
    if model_type == "SVC":
        model = SVC()
        params = {
            'model__kernel': ['linear', 'poly', 'rbf'], 
            'model__C': np.linspace(0.1, 10, 10),
            #'model__C': [2.3],
            'model__gamma': ['scale', 'auto']
        }
    else:
        raise ValueError("Unsupported model type. Use 'SVC'")

    # Define preprocessing pipeline
    ct = ColumnTransformer([
        ("dummify", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
         make_column_selector(dtype_include=object)),
        ("standardize", StandardScaler(), make_column_selector(dtype_include=np.number))
    ], remainder="passthrough").set_output(transform="pandas")

    # Build pipeline with column transformer and model
    pipeline = Pipeline([
        ('preprocessor', ct),
        ('model', model)
    ])

    # Perform Grid Search for hyperparameter tuning
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best model and best parameters
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"Best Parameters: {best_params}")

    # Cross-validation accuracy on training data
    cv_accuracy = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy').mean()
    print(f"Cross-validated Accuracy (Training): {cv_accuracy:.4f}")

    # Test set evaluation
    y_pred_test = best_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    cm = confusion_matrix(y_test, y_pred_test)
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print("Class Order:", best_model.classes_)
    print("Confusion Matrix:")
    print(cm)

    # Refit the best model on the entire dataset
    best_model.fit(X, y)
    print("Best model refit on the entire dataset.")

    return best_model


In [606]:
final_model_fit2 = get_metrics(X, y, model_type="SVC")

Best Parameters: {'model__C': np.float64(0.1), 'model__gamma': 'scale', 'model__kernel': 'linear'}
Cross-validated Accuracy (Training): 0.6074
Test Accuracy: 0.4412
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[5 6 1]
 [4 3 4]
 [2 2 7]]
Best model refit on the entire dataset.
Best Parameters: {'model__C': np.float64(0.1), 'model__gamma': 'scale', 'model__kernel': 'linear'}
Cross-validated Accuracy (Training): 0.6074
Test Accuracy: 0.4412
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[5 6 1]
 [4 3 4]
 [2 2 7]]
Best model refit on the entire dataset.


In [607]:
final_predictions2 = pd.DataFrame(
    {"id_num": test_data['id_num'],
    "political_affiliation_predicted": final_model_fit2.predict(test_data)}
)

final_predictions2.to_csv('final_predictions2.csv', index = False)

In [643]:
final_predictions2[["political_affiliation_predicted"]].value_counts(normalize=True)

political_affiliation_predicted
Independent                        0.391566
Democrat                           0.355422
Republican                         0.253012
Name: proportion, dtype: float64

In [621]:
def get_metrics(X, y, model_type):

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

    # Initialize model and params
    if model_type == "SVC":
        model = SVC()
        params = {
            'model__kernel': ['linear', 'poly', 'rbf'], 
            #'model__C': np.linspace(0.1, 10, 10),
            'model__C': [0.1],
            'model__gamma': ['scale', 'auto']
        }
    else:
        raise ValueError("Unsupported model type. Use 'SVC'")

    # Define preprocessing pipeline
    ct = ColumnTransformer([
        ("dummify", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
         make_column_selector(dtype_include=object)),
        ("standardize", StandardScaler(), make_column_selector(dtype_include=np.number))
    ], remainder="passthrough").set_output(transform="pandas")

    # Build pipeline with column transformer and model
    pipeline = Pipeline([
        ('preprocessor', ct),
        ('model', model)
    ])

    # Perform Grid Search for hyperparameter tuning
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best model and best parameters
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"Best Parameters: {best_params}")

    # Cross-validation accuracy on training data
    cv_accuracy = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy').mean()
    print(f"Cross-validated Accuracy (Training): {cv_accuracy:.4f}")

    # Test set evaluation
    y_pred_test = best_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    cm = confusion_matrix(y_test, y_pred_test)
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print("Class Order:", best_model.classes_)
    print("Confusion Matrix:")
    print(cm)

    # Refit the best model on the entire dataset
    best_model.fit(X, y)
    print("Best model refit on the entire dataset.")

    return best_model


In [639]:
final_model_fit3 = get_metrics(X, y, model_type="SVC")

Best Parameters: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Cross-validated Accuracy (Training): 0.4889
Test Accuracy: 0.7059
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[7 3 2]
 [1 9 1]
 [2 1 8]]
Best model refit on the entire dataset.
Best Parameters: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Cross-validated Accuracy (Training): 0.4889
Test Accuracy: 0.7059
Class Order: ['Democrat' 'Independent' 'Republican']
Confusion Matrix:
[[7 3 2]
 [1 9 1]
 [2 1 8]]
Best model refit on the entire dataset.


In [647]:
final_predictions3 = pd.DataFrame(
    {"id_num": test_data['id_num'],
    "political_affiliation_predicted": final_model_fit3.predict(test_data)}
)

final_predictions3.to_csv('final_predictions3.csv', index = False)